In [1]:
import numpy as np
#%matplotlib notebook
import matplotlib.pyplot as plt
#import sympy as sy
import os
import pandas as pd
from scipy.stats import pearsonr
from scipy.stats import kendalltau
from scipy.stats import spearmanr
from scipy.stats import normaltest
from scipy.stats import linregress
pd.set_option('display.max_rows', 20)
from numpy.polynomial.polynomial import polyfit
plt.rcParams['figure.dpi'] = 100
plt.rcParams['lines.markersize'] = 4
plt.rcParams['font.size'] = 18
plt.rcParams['axes.prop_cycle'] = plt.cycler('color', 'brgmyk')
#plt.rcParams["font.family"] = "Times New Roman"
from matplotlib.ticker import FormatStrFormatter

## Load data from files

In [2]:
def load_gas_files(location, 
                   altitudes=[68, 46, 32, 22], 
                   base_paths = {
                       'n2o': r'C:\Users\polgaste\Desktop\iasi_integration\mls_v6_test\correlations\data_n2o',
                       'o3':  r'C:\Users\polgaste\Desktop\iasi_integration\mls_v6_test\correlations\data_o3'
                   },
                   has_header=False):
    """
    Load N2O and O3 data files for a given location and pressure altitudes.
    
    This function assumes:
      - Files are stored in subdirectories by altitude (e.g., './n2o/68/').
      - Each file is named as "gas_location_pressure.csv", for example, 
        "n2o_quistococha_68.csv".
      - Each file contains two columns:
            1. A date column (formatted like 2018311.0)
            2. A gas concentration column.
      - The columns are separated by one or more whitespace characters.
      - If the file does not have a header row, it assigns column names "date" and "concentration".
    
    Parameters:
        location (str): The location name (e.g., 'quistococha').
        altitudes (list of int, optional): List of pressure altitudes to load. Defaults to [68, 46, 32, 22].
        base_paths (dict, optional): Dictionary mapping gas names to their base directories.
                                     Defaults to {'n2o': './n2o', 'o3': './o3'}.
        has_header (bool, optional): Whether the CSV files include a header row. Defaults to False.
        
    Returns:
        dict: A nested dictionary where the first-level keys are the gas names and the
              second-level keys are the pressure altitudes. Each leaf is a pandas DataFrame.
              For example, result['n2o'][68] contains the DataFrame for 'n2o_quistococha_68.csv'.
    """
    data = {}
    
    for gas, folder in base_paths.items():
        data[gas] = {}
        for pressure in altitudes:
            filename = f"{gas}avg_{location}_{pressure}.csv"
            # Construct the path assuming files are in a subfolder named by the pressure altitude.
            file_path = os.path.join(folder, str(pressure), filename)
            try:
                df = pd.read_csv(file_path, sep=',', header=0 if has_header else None,
                                 names=["date", "concentration"], usecols=[0, 1])
                data[gas][pressure] = df
            except FileNotFoundError:
                print(f"File not found: {file_path}")
                data[gas][pressure] = None
                
    return data

def combine_all(gas_data):
    """
    For each altitude in the gas_data dictionary, combine the N2O and O3 data by performing
    an inner join on the 'date' column. The function assumes that each DataFrame has at least
    the columns 'date' and 'concentration'.
    
    Parameters:
        gas_data (dict): A dictionary with keys 'n2o' and 'o3'. Each of these is a dictionary 
                         keyed by altitude, where the value is a DataFrame.
                         
    Returns:
        dict: A dictionary keyed by altitude. Each value is a combined DataFrame with columns:
              'date', 'n2o', and 'o3'.
    """
    combined_data = {}
    
    # Loop over altitudes (assuming both gases have the same altitude keys)
    for altitude in gas_data['n2o']:
        df_n2o = gas_data['n2o'][altitude]
        df_o3  = gas_data['o3'][altitude]
        
        if df_n2o is None or df_o3 is None:
            print(f"Skipping altitude {altitude} hPa because one of the data sets is missing.")
            combined_data[altitude] = None
            continue
        
        # Check that both DataFrames have a 'date' column
        if 'date' not in df_n2o.columns:
            print(f"Altitude {altitude} hPa: 'date' column missing in N2O data. Columns found: {df_n2o.columns}")
            combined_data[altitude] = None
            continue
        if 'date' not in df_o3.columns:
            print(f"Altitude {altitude} hPa: 'date' column missing in O3 data. Columns found: {df_o3.columns}")
            combined_data[altitude] = None
            continue
        
        # Merge on 'date' (inner join keeps only rows with shared dates)
        combined_df = pd.merge(df_n2o, df_o3, on='date', suffixes=('_n2o', '_o3'))
        # Rename columns to be clear
        combined_df.rename(columns={'concentration_n2o': 'n2o',
                                    'concentration_o3': 'o3'}, inplace=True)
        
        combined_data[altitude] = combined_df
        
    return combined_data

In [ ]:
def decimal_year(d):
    """MLS dates look like 2018311.0 (YYYYDDD)."""
    d = int(d)
    return d // 1000 + (d % 1000 - 0.5) / 365.25

def detrend_keep_mean(t, y):
    """Remove a least-squares linear trend in time; keep the series mean."""
    slope, intercept = np.polyfit(t, y, 1)
    return y - (slope * t + intercept) + y.mean()

def combine_all(gas_data, detrend=True):
    """Inner-join N2O and O3 on date for each altitude; optionally detrend both gases."""
    combined_data = {}
    for altitude in gas_data['n2o']:
        df_n2o, df_o3 = gas_data['n2o'][altitude], gas_data['o3'][altitude]
        if df_n2o is None or df_o3 is None:
            print(f"Skipping altitude {altitude} hPa because one of the data sets is missing.")
            combined_data[altitude] = None
            continue
        df = (pd.merge(df_n2o, df_o3, on='date', suffixes=('_n2o', '_o3'))
                .rename(columns={'concentration_n2o': 'n2o', 'concentration_o3': 'o3'})
                .dropna(subset=['n2o', 'o3']))
        if detrend and len(df) > 2:
            t = df['date'].map(decimal_year).to_numpy()
            for gas in ('n2o', 'o3'):
                df[gas] = detrend_keep_mean(t, df[gas].to_numpy())
        combined_data[altitude] = df
    return combined_data

In [4]:
def calculate_correlations(locations, 
                           altitudes=[68, 46, 32, 22],
                           base_paths={'n2o': r'C:\Users\polgaste\Desktop\iasi_integration\mls_v6_test\correlations\data_n2o',
                                       'o3':  r'C:\Users\polgaste\Desktop\iasi_integration\mls_v6_test\correlations\data_o3'},
                           has_header=False):
    """
    Calculate and print the correlation coefficients between N₂O and O₃ 
    for a list of locations. For each location, data is loaded from CSV files,
    combined so that only days with measurements for both gases are retained, 
    and the correlation coefficient is calculated for each altitude.

    Parameters:
      - locations (list of str): List of location names used in the file names 
                                 (e.g., ['quistococha', 'another_location']).
      - altitudes (list of int, optional): List of pressure altitudes (default: [68, 46, 32, 22]).
      - base_paths (dict, optional): Dictionary mapping gas names to their base directories.
                                     Defaults to {'n2o': './n2o', 'o3': './o3'}.
      - has_header (bool, optional): Whether the CSV files include a header row (default: False).

    Returns:
      dict: A nested dictionary with locations as the top-level keys and inner dictionaries
            that have altitudes as keys and the corresponding correlation coefficients as values.
            If the correlation cannot be computed (due to missing or insufficient data), the value is None.
    """
    correlations = {}
    
    for loc in locations:
        # Load the data for the given location.
        gas_data = load_gas_files(loc, altitudes=altitudes, base_paths=base_paths, has_header=has_header)
        # Combine the data so that only days with both measurements remain.
        combined_data = combine_all(gas_data)
        
        correlations[loc] = {}
        for altitude in altitudes:
            df = combined_data.get(altitude)
            if df is not None and len(df) > 1:
                # Calculate the Pearson correlation between N₂O and O₃.
                corr_val = df['n2o'].corr(df['o3'])
                correlations[loc][altitude] = corr_val
            else:
                correlations[loc][altitude] = None
                
    return correlations


In [5]:
# Define a mapping from internal location names to reader-friendly names.
locations_dict = {
    'bashkortostan': 'Bashkortostan',
    'bozeman': 'Montana',
    'brunei': 'Kalimantan',
    'california': 'California',
    'catalonia': 'Catalonia',
    'colombia': 'Colombia',
    'estonia': 'Estonia',
    'finland': 'Finland',
    'florianopolis': 'Santa Catarina',
    'florida': 'Florida',
    'france': 'France',
    'french_guiana': 'French Guiana',
    'huntingdon': 'Southern Québec',
    'iceland_e': 'Iceland (E)',
    'iceland_w': 'Iceland (W)',
    'khabarovsk': 'Khabarovsk',
    'kongo': 'Congo',
    'kyrgyzstan': 'Kyrgyzstan',
    'mexico': 'Xochimilco, Mexico City',
    'morocco': 'Morocco',
    'mukhrino': 'Mukhrino',
    'myanmar': 'Myanmar',
    'nz_n': 'New Zealand (N)',
    'nz_s': 'New Zealand (S)',
    'pantanal': 'Pantanal',
    'quistococha': 'Peruvian Amazon',
    'romania': 'Romania',
    'taiwan': 'Taiwan',
    'tarapoto': 'Tarapoto',
    'tasmania': 'Tasmania',
    'tierra_del_fuego': 'Tierra del Fuego',
    'uganda_e': 'Uganda (E)',
    'uganda_n': 'Uganda (N)',
    'uganda_s': 'Uganda (S)',
    'wales': 'Wales'
    # Add more mappings as needed.
}

In [6]:
# Example usage:
if __name__ == '__main__':
    # List of locations (as used in your file names)
    locations = locations_dict.keys()
    
    # Calculate correlation coefficients for each location and altitude.
    corr_results = calculate_correlations(locations, 
                                          altitudes=[68, 46, 32, 22],
                                          has_header=False)
    
    # Print the correlation results.
    for loc, alt_dict in corr_results.items():
        print(f"\nCorrelation coefficients for {loc}:")
        for altitude, corr_val in alt_dict.items():
            if corr_val is not None:
                print(f"  {altitude} hPa: {corr_val:.3f}")
            else:
                print(f"  {altitude} hPa: Not enough data to compute correlation")


Correlation coefficients for bashkortostan:
  68 hPa: -0.616
  46 hPa: -0.107
  32 hPa: 0.315
  22 hPa: 0.649

Correlation coefficients for bozeman:
  68 hPa: -0.593
  46 hPa: -0.275
  32 hPa: 0.198
  22 hPa: 0.553

Correlation coefficients for brunei:
  68 hPa: -0.266
  46 hPa: -0.504
  32 hPa: -0.512
  22 hPa: -0.590

Correlation coefficients for california:
  68 hPa: -0.556
  46 hPa: -0.443
  32 hPa: -0.107
  22 hPa: 0.480

Correlation coefficients for catalonia:
  68 hPa: -0.617
  46 hPa: -0.310
  32 hPa: 0.136
  22 hPa: 0.479

Correlation coefficients for colombia:
  68 hPa: -0.231
  46 hPa: -0.462
  32 hPa: -0.487
  22 hPa: -0.509

Correlation coefficients for estonia:
  68 hPa: -0.637
  46 hPa: -0.141
  32 hPa: 0.362
  22 hPa: 0.660

Correlation coefficients for finland:
  68 hPa: -0.522
  46 hPa: 0.008
  32 hPa: 0.352
  22 hPa: 0.524

Correlation coefficients for florianopolis:
  68 hPa: -0.488
  46 hPa: -0.633
  32 hPa: -0.350
  22 hPa: 0.598

Correlation coefficients for flo

## Correlations with p value

In [7]:
def calculate_correlations_with_p(locations, 
                                   altitudes=[68, 46, 32, 22],
                                   base_paths={'n2o': r'C:\Users\polgaste\Desktop\iasi_integration\mls_v6_test\correlations\data_n2o',
                                               'o3':  r'C:\Users\polgaste\Desktop\iasi_integration\mls_v6_test\correlations\data_o3'},
                                   has_header=False):
    """
    For each location, load the gas data, combine it so that only days with both N₂O and O₃ measurements remain,
    and then calculate the Pearson correlation coefficient and p-value for each altitude using scipy.stats.pearsonr.
    
    Parameters:
      - locations (list of str): List of location names used in the file names.
      - altitudes (list of int, optional): List of pressure altitudes (default: [68, 46, 32, 22]).
      - base_paths (dict, optional): Dictionary mapping gas names to their base directories.
      - has_header (bool, optional): Whether the CSV files include a header row.
      
    Returns:
      dict: A nested dictionary with top-level keys as locations. For each location, the inner dictionary
            has altitudes as keys and a dictionary with keys 'r' (the correlation coefficient) and 'p' (the p-value)
            as values. If the correlation cannot be computed, both are set to None.
    """
    correlations = {}
    
    for loc in locations:
        # Load the data for the given location.
        gas_data = load_gas_files(loc, altitudes=altitudes, base_paths=base_paths, has_header=has_header)
        # Combine the data to retain only rows with both measurements.
        combined_data = combine_all(gas_data)
        
        correlations[loc] = {}
        for altitude in altitudes:
            df = combined_data.get(altitude)
            if df is not None and len(df) > 1:
                try:
                    # Compute Pearson correlation and p-value.
                    r, p = pearsonr(df['n2o'], df['o3'])
                    correlations[loc][altitude] = {'r': r**2, 'p': p}
                except Exception as e:
                    correlations[loc][altitude] = {'r': None, 'p': None}
            else:
                correlations[loc][altitude] = {'r': None, 'p': None}
                
    return correlations



In [8]:
# Example usage:
if __name__ == '__main__':
    # List of locations (as used in your file names)
    locations = locations_dict.keys()
    
    # Calculate correlation coefficients for each location and altitude.
    corr_results = calculate_correlations_with_p(locations, 
                                          altitudes=[68, 46, 32, 22],
                                          has_header=False)
    
    # Print the results.
    for loc, alt_dict in corr_results.items():
        print(f"\nCorrelation coefficients for {loc}:")
        for altitude, values in alt_dict.items():
            r = values['r']
            p = values['p']
            if r is not None:
                print(f"  {altitude} hPa: r = {r:.3f}, p = {p:.5f}")
            else:
                print(f"  {altitude} hPa: Not enough data to compute correlation")


Correlation coefficients for bashkortostan:
  68 hPa: r = 0.380, p = 0.00000
  46 hPa: r = 0.011, p = 0.00029
  32 hPa: r = 0.099, p = 0.00000
  22 hPa: r = 0.421, p = 0.00000

Correlation coefficients for bozeman:
  68 hPa: r = 0.352, p = 0.00000
  46 hPa: r = 0.076, p = 0.00000
  32 hPa: r = 0.039, p = 0.00000
  22 hPa: r = 0.306, p = 0.00000

Correlation coefficients for brunei:
  68 hPa: r = 0.071, p = 0.00000
  46 hPa: r = 0.254, p = 0.00000
  32 hPa: r = 0.262, p = 0.00000
  22 hPa: r = 0.348, p = 0.00000

Correlation coefficients for california:
  68 hPa: r = 0.309, p = 0.00000
  46 hPa: r = 0.197, p = 0.00000
  32 hPa: r = 0.011, p = 0.00007
  22 hPa: r = 0.230, p = 0.00000

Correlation coefficients for catalonia:
  68 hPa: r = 0.381, p = 0.00000
  46 hPa: r = 0.096, p = 0.00000
  32 hPa: r = 0.019, p = 0.00000
  22 hPa: r = 0.229, p = 0.00000

Correlation coefficients for colombia:
  68 hPa: r = 0.053, p = 0.00000
  46 hPa: r = 0.214, p = 0.00000
  32 hPa: r = 0.238, p = 0.00

In [9]:
def remap_keys(data: dict, key_map: dict) -> dict:
    """Map each key in `data` to key_map[key] when present."""
    return {key_map[k]: v for k, v in data.items() if k in key_map}


#print(remap_keys(corr_results, locations_dict))  

In [10]:
corr_remap = remap_keys(corr_results, locations_dict)

In [11]:
import csv

def save_corr_table(data: dict, path: str, p_tiny_threshold: float = 1e-6):
    """
    data: {
      location: {
        altitude: {"r": <float>, "p": <float>},
        ...
      },
      ...
    }
    Writes CSV with rows:
      location, alt1_r, alt1_p, alt2_r, alt2_p, ...
    No headers. Altitudes are all unique altitudes across locations, sorted.
    Missing (location, altitude) cells are left blank.
    """

    def fmt_r(x):
        # stable, compact numeric formatting for r
        return f"{x:.6g}" if x is not None else ""

    def fmt_p(p):
        if p is None:
            return ""
        if p < p_tiny_threshold:
            return "<<0.001"
        # compact scientific/decimal depending on magnitude
        return f"{p:.6g}"

    # Collect and sort the union of all altitudes
    all_alts = sorted({alt for loc in data.values() for alt in loc.keys()})

    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        for location in sorted(data.keys()):
            row = [location]
            loc_block = data[location]
            for alt in all_alts:
                cell = loc_block.get(alt)
                if cell is None:
                    row.extend(["", ""])        # r, p placeholders
                else:
                    row.extend([fmt_r(cell.get("r")), fmt_p(cell.get("p"))])
            w.writerow(row)

In [12]:
save_corr_table(corr_remap, "correlations_v6.csv")

PermissionError: [Errno 13] Permission denied: 'correlations_v6.csv'

In [14]:
ZONES = {
    **{k: "NH" for k in ["Bashkortostan", "California", "Catalonia", "Estonia", "Finland", "France",
                         "Iceland (E)", "Iceland (W)", "Khabarovsk", "Kyrgyzstan", "Montana", "Morocco",
                         "Mukhrino", "Romania", "Southern Québec", "Wales"]},
    **{k: "Tropics" for k in ["Colombia", "Congo", "Florida", "French Guiana", "Kalimantan", "Myanmar",
                              "Pantanal", "Peruvian Amazon", "Santa Catarina", "Taiwan", "Tarapoto",
                              "Uganda (E)", "Uganda (N)", "Uganda (S)", "Xochimilco, Mexico City"]},
    **{k: "SH" for k in ["New Zealand (N)", "New Zealand (S)", "Tasmania", "Tierra del Fuego"]},
}
ALTS = [68, 46, 32, 22]


rows = []
for loc, name in locations_dict.items():
    combined = combine_all(load_gas_files(loc, altitudes=ALTS))
    row = {"Location": name, "Zone": ZONES[name]}
    for alt in ALTS:
        df = combined.get(alt)
        if df is None or len(df) < 3:
            continue
        t = df["date"].map(decimal_year).to_numpy()
        r, p = pearsonr(detrend(t, df["n2o"].to_numpy()), detrend(t, df["o3"].to_numpy()))
        row[f"{alt} r"] = round(r, 3)
        row[f"{alt} r2"] = round(r**2, 3)
        row[f"{alt} p"] = "<0.001" if p < 0.001 else f"{p:.3f}"
        row[f"{alt} n"] = len(df)
    rows.append(row)

corr_table = (pd.DataFrame(rows)
              .assign(Zone=lambda d: pd.Categorical(d["Zone"], ["NH", "Tropics", "SH"], ordered=True))
              .sort_values(["Zone", "Location"]))
corr_table.to_csv("correlations_v6.csv", index=False)
corr_table

,Location,Zone,68 r,68 r2,68 p,68 n,46 r,46 r2,46 p,46 n,32 r,32 r2,32 p,32 n,22 r,22 r2,22 p,22 n
0,Bashkortostan,NH,-0.662,0.438,<0.001,1141,-0.108,0.012,<0.001,1141,0.315,0.099,<0.001,1141,0.647,0.418,<0.001,1141
3,California,NH,-0.671,0.450,<0.001,1381,-0.479,0.229,<0.001,1381,-0.110,0.012,<0.001,1381,0.472,0.222,<0.001,1381
4,Catalonia,NH,-0.719,0.518,<0.001,1130,-0.347,0.120,<0.001,1130,0.121,0.015,<0.001,1130,0.464,0.216,<0.001,1130
6,Estonia,NH,-0.671,0.451,<0.001,1561,-0.148,0.022,<0.001,1561,0.363,0.132,<0.001,1561,0.660,0.435,<0.001,1561
7,Finland,NH,-0.559,0.312,<0.001,2307,-0.002,0.000,0.923,2307,0.350,0.123,<0.001,2307,0.524,0.274,<0.001,2307
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18,"Xochimilco, Mexico City",Tropics,-0.387,0.150,<0.001,1611,-0.569,0.324,<0.001,1611,-0.596,0.355,<0.001,1611,0.075,0.006,0.002,1611
22,New Zealand (N),SH,-0.671,0.451,<0.001,1138,-0.221,0.049,<0.001,1138,0.387,0.150,<0.001,1138,0.491,0.242,<0.001,1138
23,New Zealand (S),SH,-0.603,0.363,<0.001,1512,-0.163,0.026,<0.001,1512,0.366,0.134,<0.001,1512,0.497,0.247,<0.001,1512
29,Tasmania,SH,-0.650,0.422,<0.001,1125,-0.221,0.049,<0.001,1125,0.323,0.104,<0.001,1125,0.517,0.267,<0.001,1125
